# Experimental test 1 Result

* vllm기준으로 accuracy와 f1 score를 테스트
* few-shot테스트를 위해서 sc.Self_Consistency의 두번째 파라미터를 1, 2, 3, 4 로 변경하여 실험 수행
* 이외의 파라미터는 고정 
    * fewshot 테스트에 활용할 질문의 개수  : 60
    * 소스코드 포함여부  : 'Y'           
    * 반복횟수 : 5회                
    * 시스템프롬프트 'sys_prompt10'
    * self-consistency 횟수 : 5
    * temperature : 0.01
    * 엑셀버전 : 'ver7'
* 이후 결과에 대해서 스코어 비교 진행 


In [13]:
import sys, os
import re
import numpy as np
from sklearn import metrics
import pandas as pd



In [17]:
# /mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/experiment/run_id_1/sc_vq_result_4_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/experiment/run_id_3'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x)).astype(int)
            tmp['o_result'] = tmp['result']

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(f"value count for golen : {df['gold'].value_counts()}")
        print(f"value count for o_result : {df['o_result'].value_counts()}")
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [18]:
                        #   {
                        #         "llm_model"         : 'vq',              # llm_model
                        #         "model_ver"         : 'models--cyankiwi--Qwen3-30B-A3B-Instruct-2507-AWQ-4bit',         # model_ver
                        #         "few_shot_n"        : 4,                # few_shot_n
                        #         "test_n"            : 30,                # test_n(# of question for test)
                        #         "q_src_yn"          : 'Y',              # q_src_yn 
                        #         "iteration_num"     : 10,                # iteration num
                        #         "prompt_ver"        : 'sys_prompt10',   # prompt_ver
                        #         "sc_num"            : 5,                # sc_num
                        #         "temperature"       : 0.01,             # temperature
                        #         "excel_ver"         : 'ver7'            # excel_verion)
                        #     }

In [20]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vq', 3, 30, 'Y', 30, 'sys_prompt14', 3,  0.01, 'ver7')
print(list_)

value count for golen : gold
1    163
0     86
2     51
Name: count, dtype: int64
value count for o_result : o_result
1    138
0     87
2     75
Name: count, dtype: int64
              precision    recall  f1-score   support

           0      0.942     0.931     0.936        87
           1      0.804     0.949     0.870       138
           2      0.961     0.653     0.778        75

    accuracy                          0.870       300
   macro avg      0.902     0.845     0.862       300
weighted avg      0.883     0.870     0.866       300

vq_result_3_30_Y :  87.0
[np.float64(76.66666666666667), np.float64(80.0), np.float64(90.0), np.float64(90.0), np.float64(93.33333333333333), np.float64(90.0), np.float64(90.0), np.float64(86.66666666666667), np.float64(86.66666666666667), np.float64(86.66666666666667)]


In [7]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vq', 4, 30, 'Y', 30, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

None


In [ ]:
tt_0 = pd.read_csv('/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/experiment/run_id_1/sc_vq_result_4_30_Y_30_sys_prompt10_5_0.01_ver7_0.csv')

In [ ]:
tt_1 = pd.read_csv('/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/experiment/run_id_3/sc_vq_result_3_30_Y_30_sys_prompt14_3_0.01_ver7_3.csv')

In [ ]:
tt_1

In [ ]:
tt_1['gold'] = tt_1['answer'].apply(lambda x : re.sub(r'[^012]', '', x)).astype(int)
tt_1['o_result'] = tt_1['result']

gold_df = tt_1[['id', 'gold']].drop_duplicates()
chk_cnt = tt_1.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
chk_cnt = chk_cnt[chk_cnt['cnt'] == 3]
chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

# df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
# acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
# acc_list.append(acc)
# df = pd.concat([df, df_eval], axis =0)

In [ ]:
df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)

In [ ]:
df_eval
print((df_eval['equal_yn'].sum()/df_eval.shape[0])*100  )

In [ ]:
df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
y_true = df_eval['o_result']
y_pred = df_eval['gold']
print(metrics.classification_report(y_true, y_pred, digits=3))

In [ ]:
df_eval